In [1]:
import pandas as pd
import numpy as np


train = pd.read_csv("playground-series-s6e2/train.csv")
test = pd.read_csv("playground-series-s6e2/test.csv")

In [2]:
train.shape, test.shape

((630000, 15), (270000, 14))

In [3]:
display(train.head())

print("\n" + "="*132)
display(test.head())

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


In [4]:
display(train.info())

print("\n" + "="*132)

display(test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Hear

None


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270000 entries, 0 to 269999
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       270000 non-null  int64  
 1   Age                      270000 non-null  int64  
 2   Sex                      270000 non-null  int64  
 3   Chest pain type          270000 non-null  int64  
 4   BP                       270000 non-null  int64  
 5   Cholesterol              270000 non-null  int64  
 6   FBS over 120             270000 non-null  int64  
 7   EKG results              270000 non-null  int64  
 8   Max HR                   270000 non-null  int64  
 9   Exercise angina          270000 non-null  int64  
 10  ST depression            270000 non-null  float64
 11  Slope of ST              270000 non-null  int64  
 12  Number of vessels fluro  270000 non-null  int64  
 13  Thallium                 270000 non-null  int64  
dtypes: 

None

In [5]:
train_df = train.copy()

train_df['Heart Disease'] = (train_df['Heart Disease']=='Presence').astype(int)

x = train_df.drop(columns=['Heart Disease'])
y = train_df['Heart Disease']
x.shape, y.shape

((630000, 14), (630000,))

In [6]:
def feture_eng(x: pd.DataFrame):
    x = x.copy()
    if 'id' in x.columns:
        x = x.drop(columns=['id'])
    x = x.drop(columns=['FBS over 120'])

    return x

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
cat_cols = ['Chest pain type', 'EKG results', 'Slope of ST', 'Thallium']

encoder_transformer = ColumnTransformer(transformers=[
    ('target_enc', TargetEncoder(random_state=42), cat_cols),
], remainder='passthrough')

preprocessor = Pipeline([
    ('feature_eng', FunctionTransformer(feture_eng)),
    ('encoder', encoder_transformer),
    ('scaler', StandardScaler())
])

In [10]:
xgb_params = {'n_estimators': 962, 'learning_rate': 0.037395158129910566, 'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.9729126220493468, 'colsample_bytree': 0.5158352066892495, 'gamma': 0.15744297565813703}
cat_params = {'iterations': 838, 'learning_rate': 0.09981859646361525, 'depth': 3, 'l2_leaf_reg': 3, 'subsample': 0.9291146360119862, 'colsample_bylevel': 0.5159046186312849, 'silent': True}
lgbm_params = {'n_estimators': 838, 'learning_rate': 0.09981859646361525, 'num_leaves': 151, 'max_depth': 3, 'min_child_samples': 86, 'subsample': 0.9291146360119862, 'colsample_bytree': 0.5159046186312849, 'verbose': -1}

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, TargetEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [12]:
estimators = [
    ('xgb', XGBClassifier(**xgb_params, random_state=42)),
    ('cat', CatBoostClassifier(**cat_params, random_state=42)),
    ('lgbm', LGBMClassifier(**lgbm_params, random_state=42))
]

stacking_model = Pipeline([
    ('preprocessor', preprocessor),
    ('stacking', StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(),
        stack_method='predict_proba',
        cv=5,
        n_jobs=-1
    ))
])

In [13]:
X_train, X_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
stacking_model.fit(X_train, y_train)

AttributeError: The following error was raised: 'CatBoostClassifier' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.